## JSONL Asessment Files

In [1]:
import os
import pandas as pd
import json

def parse_id_list(id_list_strings):
    """
    Parses a list of strings. Handles multiple IDs on the same line.
    Example input: ['us_dsfn_cd=123 lei=456']
    Output: {'us_dsfn_cd': '123', 'lei': '456'}
    """
    id_dict = {}
    for line in id_list_strings:
        tokens = line.split()
        for token in tokens:
            token = token.strip()
            if not token: continue

            if '=' in token:
                key, val = token.split('=', 1)
                id_dict[key.strip()] = val.strip()
            else:
                id_dict[f"raw_{token}"] = token

    return id_dict

def get_split_values(raw_value):
    """
    Splits a string into (Value 1, Value 2).
    Case 1: Split by '----------//----------'
    Case 2: Split by Newline '\n'
    """
    val_str = str(raw_value).strip()
    delimiter = '----------//----------'

    if not val_str or val_str.lower() == 'nan':
        return '', ''

    if delimiter in val_str:
        parts = val_str.split(delimiter)
        v1 = parts[0].strip() if len(parts) > 0 else ''
        v2 = parts[1].strip() if len(parts) > 1 else ''
        return v1, v2
    
    elif '\n' in val_str:
        parts = val_str.split('\n', 1)
        v1 = parts[0].strip()
        v2 = parts[1].strip() if len(parts) > 1 else ''
        return v1, v2

    # 3. No separator found -> Everything belongs to Entity 1
    else:
        return val_str, ''

with open('em/data/ground_truth/ground_truth_AssessmentFiles.jsonl', 'w', encoding='utf-8') as f_out:
    folder_path = 'em/data/ground_truth/attic - resolved cases'
    delimiter = '----------//----------'
    seen_pairs = set()
    unique_entities = set()
    no_valid_assessment_count = set()
    used_files = []
    not_processed_files = []
    not_used_files = []
    assessment_count = {'y': 0, 'n': 0}

    for filename in os.listdir(folder_path):
        if not (filename.endswith('.xlsx') or filename.endswith('.xls')):
            continue
        try:
            used_files.append(filename)
            xls = pd.ExcelFile(os.path.join(folder_path, filename))
            relevant_sheets = [sheet for sheet in xls.sheet_names if sheet.startswith(('ex_post', 'Tmp-Tmp_', 'Tmp-Of', 'Tmp-vs-Of', 'Tmp-vs-Tmp'))]
            if not relevant_sheets:
                not_used_files.append(filename)
                # print(f"No relevant sheets in file {filename}, skipping.")
                continue
            for sheet_name in relevant_sheets:
                df = pd.read_excel(xls, sheet_name=sheet_name)

                unique_assessments = set(df['Assessment'].astype(str).str.lower().unique())
                if not unique_assessments.intersection({'y', 'n', 'yes', 'no'}):
                    no_valid_assessment_count.add(filename)
                    # print(f"Skipping {sheet_name} form file {filename} (No valid assessments found)")
                    continue

                for index, row in df.iterrows():
                    if sheet_name.startswith('ex_post'):
                        score = 1
                    else:
                        score = str(row.get('evidence', row.get('score', '')))
                    
                    assessment = str(row.get('Assessment', '')).strip().lower()
                    if assessment == 'yes':
                        assessment = 'y'
                    elif assessment == 'no':
                        assessment = 'n'
                    if assessment not in ('y', 'n'):
                        # print(f"Invalid assessment value '{assessment}' in file {filename}, sheet {sheet_name}, row {index}, skipping.")
                        continue
                    assessment_count[assessment] += 1

                    r1 = str(row.get('entty_riad_cd_1', '')).strip()
                    r2 = str(row.get('entty_riad_cd_2', '')).strip()

                    # Check if already seen
                    pair_signature = tuple(sorted([r1, r2]))
                    if pair_signature in seen_pairs:
                        # print(f"Duplicate pair found: {pair_signature}, skipping.")
                        continue
                        
                    seen_pairs.add(pair_signature)
                    unique_entities.add(r1)
                    unique_entities.add(r2)

                    nm_1, nm_2 = get_split_values(row.get('nm_entty_cmp', ''))
                    strt_1, strt_2 = get_split_values(row.get('strt_cmp', ''))
                    pstl_1, pstl_2 = get_split_values(row.get('pstl_cd_cmp', ''))
                    cty_1, cty_2 = get_split_values(row.get('cty_cmp', ''))

                    # 2. IDs
                    id_raw = str(row.get('id_cmp', ''))
                    if delimiter in id_raw:
                        raw_parts = id_raw.split(delimiter)
                        # entity 1
                        block_1 = raw_parts[0].strip()
                        list_1 = [block_1] if block_1 else []
                        
                        # entity 2
                        block_2 = raw_parts[1].strip() if len(raw_parts) > 1 else ''
                        list_2 = [block_2] if block_2 else []
                    
                    elif '\n' in id_raw:
                            parts = id_raw.split('\n', 1)
                            list_1 = [parts[0].strip()] if parts[0].strip() else []
                            list_2 = [parts[1].strip()] if len(parts) > 1 and parts[1].strip() else []
                    else:
                        # No separator found -> everything goes to ID_1
                        list_1 = [id_raw.strip()] if id_raw.strip() else []
                        list_2 = []

                    # Parse lists (now handles spaces correctly)
                    dict_id_1 = parse_id_list(list_1)
                    dict_id_2 = parse_id_list(list_2)
                    
                    result_json = {
                        "RIAD_CD_1": r1,
                        "RIAD_CD_2": r2,
                        "NM_ENTTY_1": nm_1,
                        "NM_ENTTY_2": nm_2,
                        "ID_1": dict_id_1,
                        "ID_2": dict_id_2,
                        "STRT_1": strt_1,
                        "STRT_2": strt_2,
                        "PSTL_CD_1": pstl_1,
                        "PSTL_CD_2": pstl_2,
                        "CTY_1": cty_1,
                        "CTY_2": cty_2,
                        "SCORE": score,
                        "ASSESSMENT": assessment
                    }

                    f_out.write(json.dumps(result_json, ensure_ascii=False) + '\n')

            used_files.append(filename)

        except Exception as e:
            not_processed_files.append(filename)
            print(f"Error processing file {filename}: {e}")

print("-" * 30)
print(f"Files Processed:                                  {len(used_files)}")
print(f"Files Not Processed:                              {len(not_processed_files)}")
print(f"Files processed but not used (wrong sheet names): {len(not_used_files)}")
print(f"Sheets Skipped (No Valid Assessment):             {len(no_valid_assessment_count)}")
print(f"Total Unique Cases (Pairs):                       {len(seen_pairs)}")
print(f"Total Distinct Entities:                          {len(unique_entities)}")
print(f"Assessment Counts:                                {assessment_count}")
print(f"Ratio y:                                          {assessment_count['y'] / (assessment_count['y'] + assessment_count['n']) * 100:.2f}%")
print(f"Ratio n:                                          {assessment_count['n'] / (assessment_count['y'] + assessment_count['n']) * 100:.2f}%")
print("-" * 30)

Error processing file Assessment file -TOP20-2020-07-03.xlsx: 'Assessment'
Error processing file Assessment file -Alice 2 -2022-06.xlsx: 'Assessment'
Error processing file Assessment file - ANAC- Elias 1 -2023-10.xlsx: 'Assessment'
------------------------------
Files Processed:                                  1031
Files Not Processed:                              3
Files processed but not used (wrong sheet names): 0
Sheets Skipped (No Valid Assessment):             230
Total Unique Cases (Pairs):                       112004
Total Distinct Entities:                          75560
Assessment Counts:                                {'y': 13719, 'n': 104273}
Ratio y:                                          11.63%
Ratio n:                                          88.37%
------------------------------


## JSONL Second Query True Positives from past DQE

In [2]:
import pandas as pd
import json
import re

def parse_identifiers_to_dict(text):
    if pd.isna(text) or text == "" or str(text).lower() == "nan" or text == "{}":
        return {}
    
    text = str(text).replace('&quot;', '"').strip()
    
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pairs = re.findall(r'"([^"]+)":\s*"([^"]+)"', text)
        result = {}
        for k, v in pairs:
            if k in result:
                if isinstance(result[k], list):
                    result[k].append(v)
                else:
                    result[k] = [result[k], v]
            else:
                result[k] = v
        return result

def clean_val(val):
    if pd.isna(val) or str(val).lower() == "nan":
        return ""
    return str(val).strip().lower()

df = pd.read_excel("em/data/ground_truth/true_positives_DQE.xlsx")
output_file = "em/data/ground_truth/true_positives_DQE.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for _, row in df.iterrows():
        line_dict = {
            "RIAD_CD_1": str(row['riad_cd_1']),
            "RIAD_CD_2": str(row['riad_cd_2']),
            "NM_ENTTY_1": clean_val(row['nm_entty_1']),
            "NM_ENTTY_2": clean_val(row['nm_entty_2']),
            "ID_1": parse_identifiers_to_dict(row['identifiers_1']),
            "ID_2": parse_identifiers_to_dict(row['identifiers_2']),
            "STRT_1": clean_val(row['strt_1']),
            "STRT_2": clean_val(row['strt_2']),
            "PSTL_CD_1": clean_val(row.get('pstl_cd_1', "")),
            "PSTL_CD_2": clean_val(row.get('pstl_cd_2', "")),
            "CTY_1": clean_val(row['cty_1']),
            "CTY_2": clean_val(row['cty_2']),
            "ASSESSMENT": 'y',
            "SCORE": 1
        }
        f.write(json.dumps(line_dict, ensure_ascii=False) + "\n")

print(f"File processed: {output_file}")

File processed: em/data/ground_truth/true_positives_DQE.jsonl


In [3]:
import json
import random
import pandas as pd

def join_jsonls(file_list, output_file):
    all_valid_lines = []
    seen_pairs = set()
    shared_pairs = set()

    for file_name in file_list:
        try:
            with open(file_name, 'r', encoding='utf-8') as f:
                file_added_count = 0
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        data = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                        
                    r1 = str(data.get('RIAD_CD_1', '')).strip()
                    r2 = str(data.get('RIAD_CD_2', '')).strip()
                    pair_signature = tuple(sorted([r1, r2]))

                    # Deduplication check - exclude duplicate pairs
                    if pair_signature in seen_pairs:
                        shared_pairs.add(pair_signature)
                        continue
                    
                    seen_pairs.add(pair_signature)
                    all_valid_lines.append(line)
                    file_added_count += 1
                    
                print(f"  > {file_name}: {file_added_count} unique new candidate pairs added.")
                
        except FileNotFoundError:
            print(f"  [ERROR] File {file_name} not found.")

    random.shuffle(all_valid_lines)
    with open(output_file, 'w', encoding='utf-8') as f:
        for line in all_valid_lines:
            f.write(line + '\n')

    df = pd.read_json(output_file, lines=True)
    total_entities = pd.concat([df['RIAD_CD_1'], df['RIAD_CD_2']]).nunique()
    
    assessment_counts = df['ASSESSMENT'].value_counts()
    assessment_ratios = df['ASSESSMENT'].value_counts(normalize=True) * 100

    print("\n" + "=" * 45)
    print("             FINAL GROUND TRUTH          ")
    print("=" * 45)
    print(f"Shared candidate pairs:             {len(shared_pairs)}") 
    print(f"Total candidate pairs:              {len(df)}")
    print(f"Total distinct entities:            {total_entities}")
    print("-" * 45)
    print("Assessment:")
    for val in assessment_counts.index:
        count = assessment_counts[val]
        ratio = assessment_ratios[val]
        print(f"{val:<4} {count:>8} ({ratio:.2f} %)")
    print("=" * 45)

input_filenames = [
    "em/data/ground_truth/ground_truth_AssessmentFiles.jsonl", 
    "em/data/ground_truth/true_positives_DQE.jsonl"
]
output_file = "em/data/ground_truth/final_ground_truth.jsonl"

join_jsonls(input_filenames, output_file)

  > em/data/ground_truth/ground_truth_AssessmentFiles.jsonl: 112004 unique new candidate pairs added.
  > em/data/ground_truth/true_positives_DQE.jsonl: 4745 unique new candidate pairs added.

             FINAL GROUND TRUTH          
Shared candidate pairs:             12
Total candidate pairs:              116749
Total distinct entities:            83236
---------------------------------------------
Assessment:
n      101738 (87.14 %)
y       15011 (12.86 %)


## Balance dataset (Threshold = 0.7)

In [4]:
import json
import random
import pandas as pd

def filter_easy_negatives(input_file, output_file, jw_threshold):
    valid_lines = []
    dropped_easy_negatives = 0
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                except json.JSONDecodeError:
                    continue
                
                assessment = str(data.get('ASSESSMENT', '')).strip().lower()
                jw_score = float(data.get('SCORE', 1.0))

                # Drop the easy negatives
                if assessment == 'n' and jw_score < jw_threshold:
                    dropped_easy_negatives += 1
                    continue
                valid_lines.append(line)
                
    except FileNotFoundError:
        print(f"[ERROR] File {input_file} not found.")
        return
        
    random.shuffle(valid_lines)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for line in valid_lines:
            f.write(line + '\n')

    df = pd.read_json(output_file, lines=True)
    total_entities = pd.concat([df['RIAD_CD_1'], df['RIAD_CD_2']]).nunique()
    assessment_counts = df['ASSESSMENT'].value_counts()
    assessment_ratios = df['ASSESSMENT'].value_counts(normalize=True) * 100

    print("\n" + "=" * 45)
    print("      BALANCED GROUND TRUTH (POST-FILTER)    ")
    print("=" * 45)
    print(f"Dropped Easy Negatives:       {dropped_easy_negatives}")
    print(f"Total candidate pairs:        {len(df)}")
    print(f"Total distinct entities:      {total_entities}")
    print("-" * 45)
    print("Assessment (New Balance):")
    for val in assessment_counts.index:
        count = assessment_counts[val]
        ratio = assessment_ratios[val]
        print(f"{val:<4} {count:>8} ({ratio:.2f} %)")
    print("=" * 45)

input_file = "em/data/ground_truth/final_ground_truth.jsonl"
output_file = "em/data/ground_truth/balanced_ground_truth.jsonl"

filter_easy_negatives(input_file, output_file, jw_threshold=0.70)


      BALANCED GROUND TRUTH (POST-FILTER)    
Dropped Easy Negatives:       63370
Total candidate pairs:        53379
Total distinct entities:      50856
---------------------------------------------
Assessment (New Balance):
n       38368 (71.88 %)
y       15011 (28.12 %)


## Create train/val/test datasets

In [5]:
import pandas as pd
import json
import os
from sklearn.model_selection import train_test_split

# 1. Configuración
input_jsonl = 'em/data/ground_truth/balanced_ground_truth.jsonl' 
dataset_name = 'riad_entities'
output_dir = f'em/data/raw/ditto_files/{dataset_name}'

os.makedirs(output_dir, exist_ok=True)

COLUMNS_MAPPING = [
    ('NM_ENTTY', 'name'),
    ('STRT', 'street'),
    ('PSTL_CD', 'postal_code'),
    ('CTY', 'city'),
    # ('RIAD_CD', 'riad_id'),
    ('ID', 'ids_concat')
]

def serialize_entity(row, side):
    """
    Serialize each entity taking multiple IDs in the same field.
    """
    suffix = f"_{side}"
    tokens = []
    
    for original_base, target_name in COLUMNS_MAPPING:
        original_key = f"{original_base}{suffix}"
        
        # Default
        final_val_str = ""
        
        # ID field could be a dictionary
        if original_base == 'ID':
            id_data = row.get(original_key)
            if isinstance(id_data, dict):
                # We gather all attributes
                valid_ids = []
                for k, v in id_data.items():
                    if v and str(v).lower() != 'nan':
                        clean_id = str(v).replace('\t', '').strip()
                        valid_ids.append(clean_id)
                
                # ID joinning
                if valid_ids:
                    final_val_str = " ".join(valid_ids)
            
            elif id_data and str(id_data).lower() != 'nan':
                 final_val_str = str(id_data).strip()
                 
        else:
            raw_val = row.get(original_key)
            if raw_val and str(raw_val).lower() != 'nan':
                final_val_str = str(raw_val).replace('\t', ' ').replace('\n', ' ').strip()
        
        tokens.append(f"COL {target_name} VAL {final_val_str}")

    return " ".join(tokens)

data = []
print("Processing all IDs...")

try:
    with open(input_jsonl, 'r', encoding='utf-8') as f:
        for line in f:
            record = json.loads(line)
            
            left_str = serialize_entity(record, 1)
            right_str = serialize_entity(record, 2)
            
            label = 1 if record['ASSESSMENT'] == 'y' else 0
            
            data.append([left_str, right_str, label])
except FileNotFoundError:
    print(f"Error: No se encuentra el fichero {input_jsonl}")
    exit()

df = pd.DataFrame(data, columns=['left', 'right', 'label'])

# Train / Validation / Test
print(f"Processing {len(df)} candidate pairs...")
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

def save_ditto_file(dataframe, filepath):
    with open(filepath, 'w', encoding='utf-8') as f:
        for _, row in dataframe.iterrows():
            f.write(f"{row['left']}\t{row['right']}\t{row['label']}\n")

save_ditto_file(train_df, os.path.join(output_dir, 'train.txt'))
save_ditto_file(val_df, os.path.join(output_dir, 'valid.txt'))
save_ditto_file(test_df, os.path.join(output_dir, 'test.txt'))
print('Finished')

Processing all IDs...
Processing 53379 candidate pairs...
Finished
